<a href="https://colab.research.google.com/github/deductiveclouds/Pytorch/blob/giri/extras/exercises/05_pytorch_going_modular_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. PyTorch Going Modular Exercises

Welcome to the 05. PyTorch Going Modular exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

## Resources and solutions

* These exercises/solutions are based on [section 05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.

**Solutions:**

Try to complete the code below *before* looking at these.

* See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/ijgFhMK3pp4).
* See an example [solutions notebook for these exercises on GitHub](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/solutions/05_pytorch_going_modular_exercise_solutions.ipynb).

## 1. Turn the code to get the data (from section 1. Get Data) into a Python script, such as `get_data.py`.

* When you run the script using `python get_data.py` it should check if the data already exists and skip downloading if it does.
* If the data download is successful, you should be able to access the `pizza_steak_sushi` images from the `data` directory.

In [1]:
# YOUR CODE HERE
%%writefile get_data.py

from pathlib import Path
import requests
import zipfile
import os

data_path = Path('data/')
image_path = data_path / 'pizza_steak_sushi'

if image_path.is_dir():
  print(f'Directory {image_path} already exists.')
else:
  image_path.mkdir(parents=True, exist_ok=True)
  print(f'Creating directory {image_path}.')

with open(file=data_path / 'pizza_steak_sushi.zip', mode='wb') as f:
  request = requests.get('https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip')
  f.write(request.content)
  print(f'Downloading pizza_steak_sushi.zip to {image_path}.')

with zipfile.ZipFile(file=data_path / 'pizza_steak_sushi.zip', mode='r') as zipref:
  zipref.extractall(image_path)
  print(f'Extracting pizza_steak_sushi.zip to {image_path}.')

os.remove(data_path / 'pizza_steak_sushi.zip')

Writing get_data.py


In [2]:
# Example running of get_data.py
!python get_data.py

Creating directory data/pizza_steak_sushi.
Extracting pizza_steak_sushi.zip to data/pizza_steak_sushi.


## 2. Use [Python's `argparse` module](https://docs.python.org/3/library/argparse.html) to be able to send the `train.py` custom hyperparameter values for training procedures.
* Add an argument flag for using a different:
  * Training/testing directory
  * Learning rate
  * Batch size
  * Number of epochs to train for
  * Number of hidden units in the TinyVGG model
    * Keep the default values for each of the above arguments as what they already are (as in notebook 05).
* For example, you should be able to run something similar to the following line to train a TinyVGG model with a learning rate of 0.003 and a batch size of 64 for 20 epochs: `python train.py --learning_rate 0.003 batch_size 64 num_epochs 20`.
* **Note:** Since `train.py` leverages the other scripts we created in section 05, such as, `model_builder.py`, `utils.py` and `engine.py`, you'll have to make sure they're available to use too. You can find these in the [`going_modular` folder on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/going_modular/going_modular).

In [3]:
# YOUR CODE HERE
%%writefile data_setup.py

"""
Contains code to create the train and test dataloaders.
"""
import os
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_loaders(train_dir:str, test_dir:str, transform:transforms.Compose, batch_size:int=1, num_workers:int=NUM_WORKERS):
  """Creates train and test dataloaders from train and test directory paths.
  Args:
    train_dir: Path to the training directory.
    test_dir: Path to the testing directory.
    transform: Torchvision transform to apply on the train and test data.
    batch_size: Number of samples per batch in each of the dataloaders.
    num_workers: Number of cpu cores to be exploited for parallel processing by the dataloaders.
  Returns:
    A tuple consisting of (train_dataloader, test_dataloader, class_names)
    where `class_names` represents the complete list of labels into which the
    prediction gets classified.
    Example usage:
      train_dataloader, test_dataloader, class_names = create_loaders(train_dir='data/pizza_steak_sushi/train', test_dir='data/pizza_steak_sushi/test', data_transform=data_transform, batch_size=32, num_workers=4)
  """
  train_data = ImageFolder(root=train_dir, transform=transform)
  test_data = ImageFolder(root=test_dir, transform=transform)

  train_dataloader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
  test_dataloader = DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

  return train_dataloader, test_dataloader, train_data.classes

Writing data_setup.py


In [4]:
%%writefile engine.py

"""
Contains code to train and test a given model.
"""

import torch
from tqdm.auto import tqdm

def train_step(model:torch.nn.Module, dataloader:torch.utils.data.DataLoader, loss_fn:torch.nn.Module, optimizer:torch.optim.Optimizer, device:torch.device):
  """ Trains the given model for a single epoch.
  Args:
    model: The model to be trained.
    dataloader: The dataloader the model is to be trained on.
    loss_fn: The loss function to minimize for the model.
    optimizer: The optimizer (algorithm) to conduct the minimization.
    device: The compute device; whether a `cpu` or `gpu`.
  Returns:
    A tuple consisting of the training loss and accuracy.
    Example: (0.1182, 0.5763)
  """
  model.train()
  train_loss, train_acc = 0, 0
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    train_loss += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class == y).sum().item() / len(y_pred)
  train_loss /= len(dataloader)
  train_acc /= len(dataloader)
  return train_loss, train_acc

def test_step(model:torch.nn.Module, dataloader:torch.utils.data.DataLoader, loss_fn:torch.nn.Module, device:torch.device):
  """ Tests the given model for a single epoch.
  Args:
    model: The model to be tested.
    dataloader: The dataloader the model is to be tested on.
    loss_fn: The loss function to minimize for the model.
    device: The compute device; whether a `cpu` or `gpu`.
  Returns:
    A tuple consisting of the testing loss and accuracy.
    Example: (0.1182, 0.5763)
  """
  model.eval()
  test_loss, test_acc = 0, 0
  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)
      y_pred = model(X)
      loss = loss_fn(y_pred, y)
      test_loss += loss.item()
      y_pred_labels = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
      test_acc += (y_pred_labels == y).sum().item() / len(y_pred)
  test_loss /= len(dataloader)
  test_acc /= len(dataloader)
  return test_loss, test_acc

def train(model:torch.nn.Module, train_dataloader:torch.utils.data.DataLoader, test_dataloader:torch.utils.data.DataLoader, loss_fn:torch.nn.Module, optimizer:torch.optim.Optimizer, epochs:int, device:torch.device):
  """ Encapsulates the train and test function calls on the respective dataloaders.
      Calculates and prints metrics at the end of each epoch.
  Args:
    model: The model to be trained.
    train_dataloader: The dataloader the model is to be trained on.
    test_dataloader: The dataloader the model is to be tested on.
    loss_fn: The loss function to minimize for the model.
    optimizer: The optimizer (algorithm) to conduct the minimization.
    epochs:int Specifies the number of epochs to train for.
    device: The compute device; whether a `cpu` or `gpu`.
  Return:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for
    each epoch.
    In the form: {train_loss: [...],
              train_acc: [...],
              test_loss: [...],
              test_acc: [...]}
    For example if training for epochs=2:
             {train_loss: [2.0616, 1.0537],
              train_acc: [0.3945, 0.3945],
              test_loss: [1.2641, 1.5706],
              test_acc: [0.3400, 0.2973]}
  """
  results = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model=model, dataloader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer, device=device)
    test_loss, test_acc = test_step(model=model, dataloader=test_dataloader, loss_fn=loss_fn, device=device)
    print(f'Epoch: {epoch + 1} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}')
    results['train_loss'].append(train_loss)
    results['train_acc'].append(train_acc)
    results['test_loss'].append(test_loss)
    results['test_acc'].append(test_acc)
  return results

Writing engine.py


In [5]:
%%writefile model_builder.py

"""
Contains code to instantiate the TinyVGG model.
TinyVGG link: https://poloclub.github.io/cnn-explainer/
"""

import torch
from torch import nn

class TinyVGG(nn.Module):
  """ Creates a model based on the TinyVGG architecture.
  Args:
    input_shape: An integer specifying the number of input channels.
    hidden_units: An integer specifying the number of neurons in each hidden layer.
    output_shape: An integer spacifying the number of output channels.
  Returns:
    None
  """
  def __init__(self, input_shape:int, hidden_units:int, output_shape:int):
    super().__init__()
    self.conv_block_1 = nn.Sequential(nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, stride=1, padding=0), nn.ReLU(), nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=0), nn.ReLU(), nn.MaxPool2d(kernel_size=2, stride=2))
    self.conv_block_2 = nn.Sequential(nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=0), nn.ReLU(), nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=0), nn.ReLU(), nn.MaxPool2d(kernel_size=2))
    self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(in_features=hidden_units*13*13, out_features=output_shape))

  """ Mandatory method to override for the model.
      Uses operator fusion i.e. method chaining to improve performance
      on a `gpu` if being run on one.
  """
  def forward(self, x: torch.Tensor()):
    return self.classifier(self.conv_block_2(self.conv_block_1(x)))

Writing model_builder.py


In [9]:
%%writefile utils.py

import torch
from torch import nn
import pathlib
from pathlib import Path

"""
Contains code for various helper functions during training and saving of the model.
"""

""" To persist the model in disk storage.
  Args:
    model: Instance of torch.nn.Module.
    target_dir: Path to the directory where to store the model as a string.
    model_name: Name of the file to store the model as.
  Returns:
    None
"""
def save_model(model:nn.Module, target_dir:str, model_name:str):
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True, exist_ok=True)
  assert model_name.endswith('.pth')
  model_save_path = target_dir_path / model_name
  torch.save(obj=model.state_dict(), f=model_save_path)
  print(f'Model {model_name} saved to directory {target_dir_path}.')

Overwriting utils.py


In [7]:
%%writefile train.py

import argparse
import torch
from torchvision import transforms
from data_setup import create_loaders
from model_builder import TinyVGG
from engine import train
from utils import save_model

parser = argparse.ArgumentParser(description='Parse command-line hyperparameters.')

parser.add_argument('--num_epochs', type=int, default=10, help='No of epochs to train for.')
parser.add_argument('--batch_size', type=int, default=32, help='No of samples per batch of training.')
parser.add_argument('--hidden_units', type=int, default=10, help='No of neurons in each hidden layer.')
parser.add_argument('--learning_rate', type=float, default=0.001, help='Learning rate to use with model during training.')
parser.add_argument('--train_dir', type=str, default='data/pizza_steak_sushi/train', help='Directory path for training images.')
parser.add_argument('--test_dir', type=str, default='data/pizza_steak_sushi/test', help='Directory path for testing images.')

args = parser.parse_args()

NUM_EPOCHS = args.num_epochs
BATCH_SIZE = args.batch_size
HIDDEN_UNITS = args.hidden_units
LEARNING_RATE = args.learning_rate

train_dir = args.train_dir
test_dir = args.test_dir

device = 'cuda' if torch.cuda.is_available() else 'cpu'

data_transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])

train_dataloader, test_dataloader, class_names = create_loaders(train_dir=train_dir, test_dir=test_dir, transform=data_transform, batch_size=BATCH_SIZE)

model = TinyVGG(input_shape=3, hidden_units=HIDDEN_UNITS, output_shape=len(class_names)).to(device)

loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

results = train(model=model, train_dataloader=train_dataloader, test_dataloader=test_dataloader, loss_fn=loss_fn, optimizer=optimizer, epochs=NUM_EPOCHS, device=device)

save_model(model=model, target_dir='models', model_name='TinyVGG.pth')

print(f'Model hyperparameters NUM_EPOCHS: {NUM_EPOCHS}, BATCH_SIZE: {BATCH_SIZE}, HIDDEN_UNITS: {HIDDEN_UNITS}, LEARNING_RATE: {LEARNING_RATE}, train_dir: {train_dir} and test_dir: {test_dir}.')

Writing train.py


In [10]:
# Example running of train.py
!python train.py --num_epochs 5 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | Train loss: 1.0996 | Train acc: 0.3720 | Test loss: 1.0942 | Test acc: 0.3333
 20% 1/5 [00:06<00:25,  6.31s/it]Epoch: 2 | Train loss: 1.0860 | Train acc: 0.3421 | Test loss: 1.0819 | Test acc: 0.3733
 40% 2/5 [00:12<00:19,  6.53s/it]Epoch: 3 | Train loss: 1.0594 | Train acc: 0.5008 | Test loss: 1.0479 | Test acc: 0.4533
 60% 3/5 [00:19<00:13,  6.52s/it]Epoch: 4 | Train loss: 1.0049 | Train acc: 0.5310 | Test loss: 1.0131 | Test acc: 0.4400
 80% 4/5 [00:25<00:06,  6.42s/it]Epoch: 5 | Train loss: 0.9224 | Train acc: 0.5749 | Test loss: 0.9744 | Test acc: 0.5333
100% 5/5 [00:32<00:00,  6.48s/it]
Model TinyVGG.pth saved to directory models.
Model hyperparameters NUM_EPOCHS: 5, BATCH_SIZE: 128, HIDDEN_UNITS: 128, LEARNING_RATE: 0.0003, train_dir: data/pizza_steak_sushi/train and test_dir: data/pizza_steak_sushi/test.


## 3. Create a Python script to predict (such as `predict.py`) on a target image given a file path with a saved model.

* For example, you should be able to run the command `python predict.py some_image.jpeg` and have a trained PyTorch model predict on the image and return its prediction.
* To see example prediction code, check out the [predicting on a custom image section in notebook 04](https://www.learnpytorch.io/04_pytorch_custom_datasets/#113-putting-custom-image-prediction-together-building-a-function).
* You may also have to write code to load in a trained model.

In [13]:
# YOUR CODE HERE
%%writefile predict.py

import argparse
import torch
from model_builder import TinyVGG
from torchvision.io import read_image
from torchvision.transforms import Resize

parser = argparse.ArgumentParser()
parser.add_argument('--image', type=str, help='Path to image being predicted.')
parser.add_argument('--model', type=str, default='models/TinyVGG.pth', help='Path to where the model is saved.')
args = parser.parse_args()

class_names = ['pizza', 'steak', 'sushi']

device='cuda' if torch.cuda.is_available() else 'cpu'

def load_model():
  model = TinyVGG(input_shape=3, hidden_units=128, output_shape=len(class_names)).to(device)
  model.load_state_dict(torch.load(args.model))
  return model

def predict_image():
  image = read_image(path=args.image).type(torch.float32)
  image /= 255
  data_transform = Resize(size=(64, 64))
  image = data_transform(image)
  model = load_model()
  model.eval()
  with torch.inference_mode():
    image = image.to(device)
    pred_probs = torch.softmax(model(image.unsqueeze(dim=0)), dim=1)
    pred_label_class = class_names[torch.argmax(pred_probs, dim=1)]
  print(f'Predicted: {pred_label_class} | Probability: {pred_probs.max():.4f}')

if __name__ == '__main__':
  predict_image()

Overwriting predict.py


In [14]:
# Example running of predict.py
!python predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg

/content/predict.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(args.model))
Predicted: sushi | Probability: 0.4011
